# Embed Legal Corpus with BGE-M3 (Colab GPU)

Upload `articles.jsonl` from `data/processed/` → embed → download `corpus_embeddings.npy`

In [ ]:
!pip install -q FlagEmbedding numpy tqdm

In [ ]:
from google.colab import files
import json

# Upload articles.jsonl
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

articles = []
with open(filename, 'r', encoding='utf-8') as f:
    for line in f:
        articles.append(json.loads(line))
print(f"Articles: {len(articles)}")

In [ ]:
texts = [a['text_truncated'] for a in articles]
aids = [a['aid'] for a in articles]

# Handle empty texts
for i, t in enumerate(texts):
    if not t.strip():
        texts[i] = f"Điều luật số {aids[i]}"

print(f"Ready to embed {len(texts)} texts")

In [ ]:
from FlagEmbedding import BGEM3FlagModel
import numpy as np
from tqdm import tqdm
import time

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print("Model loaded!")

In [ ]:
BATCH_SIZE = 256  # GPU can handle larger batches
MAX_LENGTH = 256

all_embeddings = []
t0 = time.time()

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i + BATCH_SIZE]
    output = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    all_embeddings.append(output['dense_vecs'])

embeddings = np.vstack(all_embeddings).astype(np.float32)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({len(texts)/elapsed:.1f} articles/sec)")
print(f"Shape: {embeddings.shape}")

In [ ]:
# Save and download
np.save('corpus_embeddings.npy', embeddings)
np.save('corpus_aids.npy', np.array(aids))

print(f"Embeddings: {embeddings.nbytes / 1024**2:.1f} MB")

files.download('corpus_embeddings.npy')
files.download('corpus_aids.npy')